# Identifier les textes évoquant le concept de république

In [17]:
import pandas as pd
import re

# TODO: aviser si vire id_orateur et utiliser id_acteur partout
df = pd.read_csv(
    "../data/interim/data_cleaning_interv_regrouped.csv", low_memory=False, dtype={"ID_orateur": str}
)
df.shape

(425561, 52)

## INTRODUIRE PRÉ-TRAITEMENT TEXTE

Pour simplifier la vie et faciliter aussi possibles perf d'un futur modèle, virer les parenthèses et balises ici pour s'économiser pas mal de choses du côté des noms de groupes

In [18]:
# FIXED: still have pb whith \xa0 (le léo du futur ne sait plus pourquoi c'était ça ?)
# Le léo de entre le passé et le futur a visiblement réglé le pb sans s'en rappeler
# (cf sans doute le traitement des balises html)

In [19]:
# nettoyage moche du texte
# # TODO: aviser si enlève crochets, ou juste […] [...], surtout des troncatures citations
# texte = re.sub(r"\[[^\]]*\]", "", texte)
# # TODO: aviser si on simplifie les caractères accentués si y a des pb ? et dans ce cas modif regex -> chiant


def nettoyer_texte(texte):
    if not isinstance(texte, str):
        return texte
    # Supprimer les balises HTML/XML
    texte = re.sub(r"<[^>]+>", "", texte)
    # Supprimer contenu entre parenthèses
    texte = re.sub(r"\([^)]*\)", "", texte)
    # Supprimer les espaces multiples
    texte = re.sub(r"\s+", " ", texte).strip()
    # uniformise pour avoir les bons apostrophes (nécessaire pour regex)
    texte = texte.replace("’", "'")
    return texte


df["Texte_clean"] = df["texte"].apply(nettoyer_texte)


In [20]:
# SI pas géré avant (devrait le faire dans le 2 sinon, mais possible perte avec nettoyage)
# Supprimer les lignes où 'Texte_clean' est manquant
df = df.dropna(subset=["Texte_clean"])

## Regex

Logique de la tentative :
- regex
- mais exclure certains termes
- mais comme les termes exclus peuvent apparaitre aussi avec les termes voulus, éviter de chainer et finir par virer des trucs qu'on aurait voulu (les idées républicaines sont menacées par Les Républicains)

In [21]:
# préparer les pays à exclure
with open("../data/raw/liste_pays_republique_stable.txt", "r", encoding="utf-8") as f:
    liste_pays = [line.strip() for line in f]

# créer un pattern regex pour les pays
# ici pas besoin d'avoir un groupe de capture par pays mais juste global ok
pattern_pays = r"(\b(?:" + r"|".join(re.escape(p) for p in liste_pays) + r")\b)"


# TODO: check avec matthias si ok
# com matthias : Regex pays insuffisante car doit aussi comprendre la forme adjectivable des pays et république en minuscule
# > ajout d'une version stable avec forme adjectivable
# > et gestion de la casse (maj/min) dans la fonction par la regex (cf re.I)

# Pour matthias : j'ai pour l'instant gardé les \b (word boundary)
# cf. je pense que ton soucis avec République d'Arménie était à cause de l'apostrophe


In [22]:
# Regex du champ lexical République (simplifié ici)

pattern_lexical = re.compile(
    r"républi",  # même au milieu des mots
    re.I,
)

# Regex des expressions à exclure

# Expressions à exclure - casse exacte
pattern_excl_case_sensitive = re.compile(
    r"\b[LlDd]es Républicains\b"  # garde la casse pour identifier le parti (et pas un adjectif)
)  # voir pour élu Républicain ? doute

# Expressions à exclure - ignorer la casse
pattern_excl_case_insensitive = re.compile(
    # partis et groupes politiques
    r"|(\bgauche démocrate et républicaine)"
    r"|(\brépublique en marche\b)"
    r"|(\bsocialiste, écologiste et républicain\b)"
    # fonctions et institutions
    r"|(\bprésident[s]? de la République\b)"
    r"|(\bprésidence[s]? de la République\b)"
    r"|(\bprocureur[s]? de la République\b)"
    r"|(\bcour[s]? de justice de la République\b)"
    r"|(\bcour[s]? de sûreté de la République\b)"
    r"|(\badministration générale de la République\b)"
    r"|(\bGouvernement de la République française\b)"
    # pays
    r"|(\brépublique[s]? soviétique[s]?\b)"  # pas un pays mais des expressions
    r"|(" + pattern_pays + ")",  # ajout des exclusions de pays si existe
    re.I,
)


def contains_lexical_outside_excl(text):
    # Trouver les positions des expressions exclues
    excl_positions = []

    # Ajouter les exclusions sensibles à la casse
    excl_positions.extend(
        [m.span() for m in pattern_excl_case_sensitive.finditer(text)]
    )

    # Ajouter les exclusions insensibles à la casse
    excl_positions.extend(
        [m.span() for m in pattern_excl_case_insensitive.finditer(text)]
    )

    # Fonction pour vérifier si une position est dans une zone exclue
    def in_excl(pos):
        for start, end in excl_positions:
            if start <= pos < end:
                return True
        return False

    # Chercher toutes les occurences du champ lexical
    for match in pattern_lexical.finditer(text):
        start_pos = match.start()
        if not in_excl(start_pos):
            return True
    return False

In [23]:
# bloc d'essai
mon_texte = "république soviétique"
contains_lexical_outside_excl(mon_texte)


False

In [24]:
# Appliquer sur la colonne
df["repu_match_valide"] = df["Texte_clean"].apply(contains_lexical_outside_excl)

In [25]:
# Exporter fichier avec 2 colonnes pour calcul avec proportions 
df.to_csv(
    "../data/interim/df_regroup_repu_proportion.csv",
    index=False,
    # quoting=csv.QUOTE_ALL,  # not needed anymore ?
)

In [26]:
df_match = df[df["repu_match_valide"]]
df_match.shape

(11429, 54)

In [27]:
# Exporter fichier uniquement avec Rep pour valeur absolues 
df_match.to_csv(
    "../data/interim/df_regroup_repu_absolu.csv",
    index=False,
    # quoting=csv.QUOTE_ALL,  # not needed anymore ?
)

In [28]:
# # verif ecriture/lecture ok
# print("df_match shape:", df_match.shape)

# df_test = pd.read_csv("../data/interim/df_repu.csv", low_memory=False)

# print("df_test shape (après export import): ", df_test.shape)

## Garder trace pour entrainement classifier bert

In [29]:
# TODO: Introduire une nouvelle colone avec juste match simple republi
# Sert à Matthias pour le classifier
# Mais corriger son truc qui a un mini pb sur le return True return False

In [30]:
# df["match_republi_simple"] = df["Texte_clean"].apply(
    # lambda x: bool(pattern_lexical.search(x))
#)

In [31]:
# df.to_csv(
   # "../data/interim/df_identification_republi_simple.csv",
   # index=False,
    # quoting=csv.QUOTE_ALL,  # not needed anymore ?
#)


In [32]:
# # verif ecriture/lecture ok
# print("df shape:", df.shape)

# df_test = pd.read_csv(
#     "../data/interim/df_identification_republi_simple.csv", low_memory=False
# )

# print("df_test shape (après export import): ", df_test.shape)